# Silver Layer — Imperative Approach

## Objective

Transform the Bronze Ingestion Delta table into two clean, typed and 
validated Silver tables ready for analytical consumption.

This notebook implements the **imperative approach** — every step 
(read, cast, deduplicate, validate, write) is explicit. A declarative 
DLT version is implemented in `04_silver_declarative`.

## Architecture

## Notebook Flow

1. **Read source**  
   Load the Delta table from Bronze Ingestion.

2. **Build `silver_dim_stock`**  
   - Select dimension attributes (`symbol`, `last_refreshed`, `time_zone`)
   - Cast `last_refreshed` to DATE
   - Deduplicate by `symbol`
   - Add `ingest_timestamp`

3. **Build `silver_fact_prices`**  
   - Select fact attributes (`symbol`, `trade_date`, OHLCV)
   - Cast numeric fields to DOUBLE / LONG
   - Cast `trade_date` to DATE
   - Deduplicate by `symbol + trade_date`
   - Apply Data Quality rules (price > 0, volume >= 0)
   - Add `ingest_timestamp`

4. **Write to ADLS Silver**  
   Persist both tables as Delta with `overwrite` mode.

## Design Decisions

- **Type casting** is performed in Silver (not Bronze) to keep raw 
  data immutable in Bronze Landing
- **Deduplication** uses natural keys — no surrogate keys needed
- **Data Quality** failures drop invalid rows silently (logged separately)
- **Overwrite mode** assumes Silver is recomputed from Bronze each run

In [0]:
import json

# Load one raw file from Bronze
raw = dbutils.fs.head(
    "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/AAPL_20260429_115810.json"
)

data = json.loads(raw)

# See top level keys
print("=== TOP LEVEL KEYS ===")
print(list(data.keys()))

# Meta Data
print("\n=== META DATA ===")
print(json.dumps(data["Meta Data"], indent=2))

# First trading day
print("\n=== FIRST TRADING DAY ===")
first_date = list(data["Time Series (Daily)"].keys())[0]
print(f"Date: {first_date}")
print(json.dumps(data["Time Series (Daily)"][first_date], indent=2))

In [0]:
df = spark.read.json('abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/')
df.printSchema()

df.show(1, truncate=False)